# 04 — Final selection, costs and statistical robustness

This notebook inspects the locked-test equity curves, transaction-cost sensitivity, bootstrap intervals and multiple-comparison-adjusted Diebold–Mariano tests.

In [1]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

ROOT = Path.cwd()
if not (ROOT / 'data').exists(): ROOT = (ROOT / '..').resolve()
processed = ROOT / 'data/processed'
summary = json.loads((ROOT / 'reports/experiment_summary.json').read_text())
display(pd.DataFrame(summary['assets']).T[['winner', 'validation_score']])
gold_equity = pd.read_csv(processed / 'gold_test_equity.csv', index_col=0, parse_dates=True)
silver_equity = pd.read_csv(processed / 'silver_test_equity.csv', index_col=0, parse_dates=True)
gold_costs = pd.read_csv(processed / 'gold_cost_sensitivity.csv')
silver_costs = pd.read_csv(processed / 'silver_cost_sensitivity.csv')

,winner,validation_score
gold,extra_trees,0.501896
silver,tree_blend,0.496938


In [2]:
sns.set_theme(style='whitegrid', context='notebook')
fig, axes = plt.subplots(1, 2, figsize=(16, 6), constrained_layout=True)
gold_equity['equity'].plot(ax=axes[0], color='#d49a00', title='Gold locked-test equity')
silver_equity['equity'].plot(ax=axes[0], color='#777777', title='Gold and Silver locked-test equity')
axes[0].set_ylabel('Growth of $1')
gold_equity['drawdown'].plot(ax=axes[1], color='#b33a3a', label='Gold')
silver_equity['drawdown'].plot(ax=axes[1], color='#555555', label='Silver')
axes[1].set_title('Drawdown')
axes[1].legend()
plt.show()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(gold_costs['transaction_cost_bps'], gold_costs['sharpe'], marker='o', label='Gold')
ax.plot(silver_costs['transaction_cost_bps'], silver_costs['sharpe'], marker='o', label='Silver')
ax.axhline(0, color='black', linewidth=0.8)
ax.set(title='Sharpe sensitivity to transaction costs', xlabel='Cost (bps per turnover unit)', ylabel='Net Sharpe')
ax.legend()
plt.show()

/var/folders/ys/hvb2c_0n7lb1bsh31xyf6d2h0000gp/T/ipykernel_88536/1475481936.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/ys/hvb2c_0n7lb1bsh31xyf6d2h0000gp/T/ipykernel_88536/1475481936.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [3]:
for asset in ['gold', 'silver']:
    tests = pd.read_csv(processed / f'{asset}_statistical_tests.csv')
    print(asset.upper())
    display(tests[['competitor', 'dm_statistic', 'p_value_holm', 'significant_5pct_holm', 'sharpe_difference', 'difference_ci_low', 'difference_ci_high']])

gold_tests = pd.read_csv(processed / 'gold_statistical_tests.csv')
plot_tests = gold_tests.copy()
plot_tests['minus_log10_holm_p'] = (-plot_tests['p_value_holm'].clip(lower=1e-300).apply(lambda x: __import__('math').log10(x)))
plt.figure(figsize=(12, 5))
sns.barplot(data=plot_tests.sort_values('minus_log10_holm_p'), x='competitor', y='minus_log10_holm_p', color='#244a7c')
plt.axhline(-__import__('math').log10(0.05), color='#b33a3a', linestyle='--', label='5% threshold')
plt.xticks(rotation=35, ha='right')
plt.title('Gold: evidence against equal forecast accuracy after Holm correction')
plt.legend()
plt.show()

GOLD


,competitor,dm_statistic,p_value_holm,significant_5pct_holm,sharpe_difference,difference_ci_low,difference_ci_high
0,zero,-2.070509,2.304285e-01,False,0.970543,-0.008254,1.790135
1,xgboost,0.590460,1.000000e+00,False,0.495792,-0.222230,1.283545
2,tree_blend,-0.126817,1.000000e+00,False,0.820437,-0.086761,1.837995
3,ridge,0.195219,1.000000e+00,False,0.847364,-0.438156,2.196288
4,hist_gradient_boosting,-1.450878,7.340687e-01,False,2.155390,1.164682,3.306803
5,elasticnet,0.557490,1.000000e+00,False,1.135179,-0.057184,2.397430
6,moving_average,-3.968534,5.062126e-04,True,0.705916,-0.555048,1.975259
7,time_mixer,-6.141458,6.541392e-09,True,2.015878,0.694502,3.503476
8,patch_tst,-19.535299,6.052463e-84,True,3.501430,2.167304,4.874598
9,last_return,-7.039725,1.733577e-11,True,2.671621,1.416978,3.939080


SILVER


,competitor,dm_statistic,p_value_holm,significant_5pct_holm,sharpe_difference,difference_ci_low,difference_ci_high
0,zero,-0.916805,1.000000e+00,False,0.630067,-0.350591,1.502270
1,patch_tst,-19.562549,3.547985e-84,True,2.111313,0.684041,3.343940
2,hist_gradient_boosting,-2.279664,1.357656e-01,False,0.370829,0.092624,0.736578
3,ridge,0.352496,1.000000e+00,False,-0.499605,-1.592154,0.598767
4,elasticnet,0.554562,1.000000e+00,False,-0.073757,-1.047971,1.005119
5,time_mixer,-5.711719,1.006566e-07,True,0.790433,-0.419067,2.081589
6,moving_average,-3.304866,6.651526e-03,True,0.479261,-1.045776,1.754518
7,extra_trees,0.876054,1.000000e+00,False,-0.125478,-0.838416,0.541818
8,xgboost,-0.637035,1.000000e+00,False,0.008570,-0.786735,0.691902
9,tsmixer,-15.103609,1.533074e-50,True,1.087209,-0.125178,2.344964


/var/folders/ys/hvb2c_0n7lb1bsh31xyf6d2h0000gp/T/ipykernel_88536/1959690510.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**How to read the plots.** Equity curves compound the signed next-day return after transaction costs; drawdown is the distance below the prior equity high. Cost sensitivity tests whether the signal survives implementation friction, while the corrected statistical table separates a robust forecast-loss difference from a noisy Sharpe ranking.